# 阶段 3：采集三类场景的特征，存成 CSV

硬件每帧统计 `sumY / sumY2 / sumGrad / pixel_cnt`，本 notebook 负责换算成
`luma / contrast / sharpness`、打标签，最后存成 `data/features.csv`。

板子上的 overlay 可能是阶段 12 / 13 的版本（`VERSION = 0x0001020D` 是阶段 13），
只要 ≥ 3 就有特征统计寄存器和 MLP，不影响采集。

## 这套流程怎么走

1. **光线自检格**：自动 / 手动各读一次 luma —— 2 分钟分辨「没光进镜头」和「寄存器没生效」；
2. **扫描格**：扫曝光 × 增益，定一个工作点（要在「常规档」光照下扫）；
3. **检查 A 格**：确认特征在动、光照分布正常；
4. **三格分开采**：label 0 / 1 / 2 各自一格，**摆好一类的光就采一类**，顺序随意。
   重复跑同一格只会覆盖那一类（不会重复累加）；
5. 最后保存 + 体检：**0 FAIL 才能进阶段 4 重训**。

三类目标中心：`f_luma ≈ 10 / 67 / 109`（= 盒子标定的三个档位）。
具体怎么摆见 `docs/04_demo_box.md` 第 3、4 节。

In [ ]:
import os
import sys
import time

import numpy as np

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib import features_io
from lib import exposure
from lib.video_setup import VIDEO_WIDTH, VIDEO_HEIGHT
from lib.ai_video_regs import AiVideoRegs, PIXEL_CNT

EXP_LINES = exposure.DEFAULT_EXPOSURE      # 曝光行数，第 1 格会重新扫
GAIN_CODE = exposure.DEFAULT_GAIN          # 增益码，同上

# 盒子模式：bring_up 一步到位就把曝光/增益锁死（见 docs/04_demo_box.md 第 6 节）
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=dict(exposure=EXP_LINES, gain=GAIN_CODE))
vs.stop_all_channels(vdma0, vdma1)
vp.start_capture(vdma0, mode)

regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X (阶段 %d)" % (regs.version(), regs.version_stage()))

# f_luma 是整数特征，乘 1.138 换回 0~255 的亮度（PIXEL_CNT 是寄存器偏移，别当像素数用）
LUMA_PER_F_LUMA = (1 << 20) / float(VIDEO_WIDTH * VIDEO_HEIGHT)


def read_luma(frames=1, settle=0.0):
    """读几帧亮度的平均值（只做人工检查；MLP 吃的仍是那三个整数特征）。"""
    vals = []
    for _ in range(frames):
        if settle:
            time.sleep(settle)
        regs.wait_new_frame()
        vals.append(regs.features()[0])
    return float(np.mean(vals))

## 第 0 步：光线自检（2 分钟）

分辨两种完全不同的故障：

* **两种模式下 luma 都是 4 左右** → 光根本没进镜头（查灯电源、调光旋钮、箱盖、镜头孔）。
  这是硬件问题，改寄存器没用；
* **自动模式明显比手动亮** → 光进得来、曝光通路正常，可以往下扫工作点。

读数前先把箱子摆成「常规档」：镜头贴近目标物（白纸 / 文字卡，占画面 60% 以上）、
主光源中亮。

In [ ]:
print("1) 自动曝光（AEC/AGC 开）：")
exposure.set_auto(iic)
time.sleep(1.0)
luma_auto = read_luma(3)
print("   luma = %.2f" % luma_auto)

print("2) 手动曝光（曝光 %d 行 / 增益 0x%02X）：" % (EXP_LINES, GAIN_CODE))
exposure.set_manual(iic, exposure=EXP_LINES, gain=GAIN_CODE)
time.sleep(0.6)
luma_manual = read_luma(3)
print("   luma = %.2f" % luma_manual)

if max(luma_auto, luma_manual) < 20:
    print()
    print("!! 两种模式下都是黑的 -> 光没进镜头。先开灯 / 查灯电源 / 开箱盖，")
    print("   再确认镜头对着的是有光的一侧。改寄存器解决不了这个问题。")
else:
    print()
    print("两种模式读数差 %.1f -> 光进得来、曝光通路正常，继续下一格扫工作点"
          % abs(luma_auto - luma_manual))
    if luma_auto < 60:
        print("提示：自动曝光也只有 %.0f，说明箱内照度偏低。" % luma_auto)
        print("      目标物再贴近一点、灯再靠近一点；还不行就得换更亮的灯。")

## 第 1 步：扫工作点（曝光 × 增益）

**必须先把箱子摆成「常规档」**（镜头贴近目标物 + 主光源中亮），否则扫出来全是噪声底。

同时扫曝光和增益：曝光上限约 736 行（受 VTS 限制），灯不够亮时光靠加曝光到不了目标。
目标是常规档 `luma ≈ 124`（`f_luma ≈ 109`），且不饱和（别到 250 以上）。

In [ ]:
EXPS = (256, 384, 512, 640, 736)      # 曝光行数（736 是实际上限）
GAINS = (0x10, 0x20, 0x40, 0x80)      # 增益码（换算关系不确定，直接扫）
TARGET_LUMA = 124.0

print("扫 %d 组，约 %.0f 秒……" % (len(EXPS) * len(GAINS), len(EXPS) * len(GAINS) * 0.7))
best = None
for g in GAINS:
    line = "  增益 0x%02X :" % g
    exposure.set_gain(iic, g)
    for e in EXPS:
        exposure.set_exposure(iic, e)
        time.sleep(0.35)
        l = read_luma(2)
        line += "  e%3d->%6.2f" % (e, l)
        if l < 250 and (best is None or abs(l - TARGET_LUMA) < abs(best[2] - TARGET_LUMA)):
            best = (g, e, l)
    print(line)

print()
if best is None:
    print("!! 全都过曝（luma>=250）：灯太亮，先把灯调暗再扫")
elif best[2] < 20:
    print("!! 最亮的一组也只有 luma=%.2f -> 画面上根本没有光。" % best[2])
    print("   先跑上一格「光线自检」确认灯开着、光能进镜头，再回来扫。")
else:
    GAIN_CODE, EXP_LINES = best[0], best[1]
    exposure.set_manual(iic, exposure=EXP_LINES, gain=GAIN_CODE)
    time.sleep(0.5)
    print("锁定工作点：曝光 %d 行 + 增益 0x%02X -> luma %.2f（目标 %.0f）"
          % (EXP_LINES, GAIN_CODE, best[2], TARGET_LUMA))
    if best[2] < 0.65 * TARGET_LUMA:
        print()
        print("!! 这个工作点偏暗（%.1f vs 目标 %.0f）：先别加增益，按这个顺序查——"
              % (best[2], TARGET_LUMA))
        print("   1) 目标物再贴近镜头一点（占画面 60% 以上）；")
        print("   2) 灯再贴近目标物、斜 30~45 度照，去掉过厚的柔光板；")
        print("   3) 都试过还上不去，才是增益/灯不够（增益同时放大噪声，能不用就不用）。")
    else:
        print("记进 docs/04_demo_box.md 的记录里，演示当天照这个复现")

## 检查 A：确认特征在动、光照分布正常

正常应该看到 luma 在几十以上、sharpness 随细节变化。拧一下灯：**luma 应该立刻跟着变**。

下面还会顺手判一下画面形状：如果出现「均值很低、对比度却很大」，
说明是一小块亮斑 + 大片黑（光没照到目标物，或者镜头直接看到了灯）。

In [ ]:
for k in range(10):
    regs.wait_new_frame()
    luma, contrast, sharpness = regs.features()      # 物理量，人工检查用
    f1, f2, f3 = regs.mlp_features()                 # MLP 真正吃的三个特征
    print("luma=%6.2f contrast=%6.2f sharp=%6.2f | f=(%3d,%6d,%3d) pixels=%d"
          % (luma, contrast, sharpness, f1, f2, f3, regs.rd(PIXEL_CNT)))
    time.sleep(0.1)
if luma < 20:
    print("!! luma 还是贴在噪声底：先解决光照，别急着采集")
elif contrast > 0.6 * luma:
    print("!! 画面是「一小块亮斑 + 大片黑」（均值 %.0f、方差 %.0f）：" % (luma, contrast))
    print("   光没照到目标物上，或者镜头直接看到了灯/光斑。")
    print("   把镜头贴近目标物、灯斜 30~45 度照目标物正面，灯本体别进画面。")
else:
    print("光照分布看着正常（均值 %.0f、对比度 %.0f，比值 %.2f）"
          % (luma, contrast, contrast / max(luma, 1e-9)))

## 采集：三类各一格，摆好一类的光就采一类

下面三格分别采 label 0 / 1 / 2，**顺序随意、可以分几次做**：
摆好某一类的光 → 跑对应那格 → 再去摆下一类。

* 每一格采 3 轮 × 60 帧（每轮换一种摆法，轮内先倒计时让你摆场景）；
* **重复跑同一格 = 覆盖那一类**（不会重复累加）；
* 每轮开始前会检查当前 `f_luma` 是否在该类的目标带里，**不在就跳过这一轮**，
  末尾会列出跳过了几轮、当前还差哪几类。

各档的目标（`f = (luma, luma2, grad)`）：

| label | 场景 | 目标 f | f_luma 目标带 | 怎么摆 |
|-------|------|--------|--------------|--------|
| 0 | 暗光 | (≈10, ≈1, ≈25) | 4~30 | 主光源全关 / 最暗档，目标物（白纸）留在画面里 |
| 1 | 逆光 | (≈67, **≈39**, ≈30) | 45~95 | 后壁光板开 + 主光源关，前面立**黑卡纸剪影**（不能用纸上的字） |
| 2 | 常规/细节 | (≈109, ≈55, **≈35**) | 80~145 | 镜头贴近白纸（占画面 60%+），灯斜 30~45° 照纸面 |

想改每类采几轮 / 每轮几帧 / 倒计时（或临时允许采「不在目标带」的轮次），
改下面那格开头的参数即可。

In [ ]:
ROUNDS = 3            # 每类采几轮（每轮换一种摆法）
PER_ROUND = 60        # 每轮采多少帧
WAIT_S = 8            # 每轮开始前的摆场景倒计时（秒）
SKIP_OUT_OF_BAND = True   # 亮度不在目标带里就跳过这一轮（保护数据）

EXPECT_BAND = {0: (4, 30), 1: (45, 95), 2: (80, 145)}      # f_luma 目标带
# f_luma2 目标带：防「亮度对了、画面形状不对」（比如常规档过曝 -> f_luma2 飙到 100）
EXPECT_BAND2 = {0: (0, 10), 1: (20, 60), 2: (30, 80)}
# f_grad 目标带：防「内容太少/太多」（暗光档没有纹理 -> f_grad 掉到 8 以下）
EXPECT_BAND3 = {0: (6, 45), 1: (15, 45), 2: (18, 55)}
CLASS_NAME = {0: "暗光", 1: "逆光", 2: "常规/细节"}
TARGET_F = {0: (10, 1, 25), 1: (67, 39, 30), 2: (109, 55, 35)}

# 每类三轮换什么摆法
HINTS = {
    0: ["主光源全关、箱盖合上（最暗的一档）",
        "主光源全关，镜头/目标物换个角度（内容变了、亮度还是暗）",
        "主光源拧到最暗档（还亮一点点）"],
    1: ["后壁光板开 + 主光源关 + 黑卡纸剪影（对比度最高）",
        "剪影换细一点的，让高光面积变大",
        "后壁光板开 + 主光源拧到最弱（半明半暗）"],
    2: ["镜头贴近白纸（占画面 60%+），灯斜 30~45 度照纸面",
        "目标物不动，灯和镜头的角度各换一点",
        "换一页字更大更粗的纸（让细节量变化）"],
}

rows = []      # 三类共用；重复跑同一类会覆盖那一类，不会累加


def status():
    have = sorted(set(int(r[3]) for r in rows))
    print("当前已有：%s ；还差：%s"
          % ("、".join(CLASS_NAME[k] for k in have) or "无",
             "、".join(CLASS_NAME[k] for k in (0, 1, 2) if k not in have) or "（三类齐了）"))


def collect_class(label):
    """单独采一类。重复运行会先清掉这一类旧数据，再采新的。"""
    global rows
    name = CLASS_NAME[label]
    lo, hi = EXPECT_BAND[label]
    print("=" * 70)
    print("采集 label %d（%s）  f_luma 目标带 %d~%d，目标 f=%s"
          % (label, name, lo, hi, TARGET_F[label]))
    print("=" * 70)

    rows = [r for r in rows if int(r[3]) != label]     # 覆盖这一类的旧数据
    got_all, skipped = [], []
    for r in range(ROUNDS):
        print()
        print("第 %d/%d 轮：%s" % (r + 1, ROUNDS, HINTS[label][r % len(HINTS[label])]))
        print("  %d 秒后开始采 %d 帧" % (WAIT_S, PER_ROUND))
        time.sleep(WAIT_S)

        f1, f2, f3 = regs.mlp_features()
        lo2, hi2 = EXPECT_BAND2[label]
        lo3, hi3 = EXPECT_BAND3[label]
        bad = []
        if not (lo <= f1 <= hi):
            bad.append("f_luma=%d 不在 %d~%d" % (f1, lo, hi))
        if not (lo2 <= f2 <= hi2):
            bad.append("f_luma2=%d 不在 %d~%d" % (f2, lo2, hi2))
        if not (lo3 <= f3 <= hi3):
            bad.append("f_grad=%d 不在 %d~%d" % (f3, lo3, hi3))
        if bad:
            msg = "；".join(bad)
            print("  !! " + msg)
            if SKIP_OUT_OF_BAND:
                print("     跳过这一轮（不采脏数据）")
                skipped.append(msg)
                continue
            print("     （SKIP_OUT_OF_BAND=False，继续采）")

        # 带通滤波：只收落在这一类目标带内的帧。
        # 实测教训：手动让场景"动起来"时画面会漂出带外（常规档漂到 f_luma=58，
        # 落进逆光区间），两类分布就糊在一起了（类间 0.6σ，MLP 学不出来）。
        # 有了它你可以放心动，只有带内的帧会被记录。
        def _accept(a, b, c, lo=lo, hi=hi, lo2=lo2, hi2=hi2, lo3=lo3, hi3=hi3):
            return (lo <= a <= hi) and (lo2 <= b <= hi2) and (lo3 <= c <= hi3)

        got = regs.collect(label, n=PER_ROUND, timeout=120.0, poll_sleep=1e-3,
                           verbose=True, accept=_accept)
        got_all += got
        dropped = getattr(regs, "last_dropped", 0)
        n_uniq = len(set(tuple(r[:3]) for r in got))
        print("  本轮 %d 帧（丢弃 %d 帧），其中 %d 个不同取值；最后一帧 f=(%d,%d,%d)"
              % (len(got), dropped, n_uniq, got[-1][0], got[-1][1], got[-1][2]))
        if dropped > PER_ROUND:
            print("     !! 丢弃的比采到的还多：画面大部分时间不在这一类的带内。")
            print("        动小一点，或者先把摆法调到位再动 —— 类内要有变化，但不能跨过类边界。")
        if n_uniq <= 2:
            print("     !! 这一轮 60 帧几乎没有变化 -> 下一轮请在采集过程中让场景动起来：")
            print("        缓慢移动/转动目标物、改一点距离或灯位。每类的不同取值越多，")
            print("        实机落在中间地带（比如 f_luma=30）时判决才越稳。")

    rows += got_all
    print()
    if got_all:
        arr = np.array([r[:3] for r in got_all], dtype=float)
        print("label %d 小结：n=%d  不同取值=%s  f均值=(%.1f, %.1f, %.1f)  约亮度=%.0f"
              % (label, len(got_all),
                 tuple(len(set(arr[:, c].tolist())) for c in range(3)),
                 arr[:, 0].mean(), arr[:, 1].mean(), arr[:, 2].mean(),
                 arr[:, 0].mean() * LUMA_PER_F_LUMA))
        print("            目标 f=%s" % (TARGET_F[label],))
        span = tuple(int(arr[:, c].max() - arr[:, c].min()) for c in range(3))
        sd = tuple(float(arr[:, c].std()) for c in range(3))
        print("            覆盖宽度=(%d, %d, %d)  标准差=(%.1f, %.1f, %.1f)" % (span + sd))
        print("            判据：覆盖宽度 >= (10, 6, 5) 让类内有变化；"
              "但 f_luma 标准差要 <= 10，否则两类的云会糊在一起")
        if sd[0] > 12:
            print("            !! f_luma 标准差 %.1f 偏大：类内变化跨过类边界了。"
                  "最近的两类中心只差 ~42，std 超过 12 就必然重叠" % sd[0])
            print("               做法：动得小一点（目标物缓慢小范围移动、换纹理），"
                  "带通滤波会帮你去掉漂出带外的帧")
    else:
        print("label %d 一帧都没采到（全被跳过）：先把这一类的光调到位" % label)
    if skipped:
        print("  被跳过的轮次：%d/%d" % (len(skipped), ROUNDS))
    status()
    return got_all

### 第 3-1 格：采集 label = 0（暗光）

**主光源全关**（或拧到最暗档），目标物留在画面里 —— 要的是「暗但有细节」，
不是「全黑」：`f_luma ≈ 10` 的同时 `f_grad` 别掉到 15 以下。

三轮的摆法见提示。跑完看小结里的 `n` 和 `不同取值`。

In [ ]:
got0 = collect_class(0)

### 第 3-2 格：采集 label = 1（逆光）

**后壁光板开、主光源关**，前面立一块**黑卡纸剪影**。

注意：逆光的判据是**对比度高**（`f_luma2 ≈ 39`，换算是 std ≈ 74），
整体亮度反而比常规档**低**（`f_luma ≈ 67`）。纸上的字是小尺度纹理，
只能抬 `f_grad`，给不了这种大尺度明暗 —— 所以这一档要用剪影，不能用纸。

In [ ]:
got1 = collect_class(1)

### 第 3-3 格：采集 label = 2（常规 / 细节）

**镜头贴近白纸（占画面 60% 以上）、主光源中亮、灯斜 30~45° 照纸面**，
灯前一层薄柔光板压光斑。目标是 `f_luma ≈ 109`、`f_grad ≈ 35`（细节最多）。

白纸上的黑字正好提供边缘，是这一档最合适的目标物。

In [ ]:
got2 = collect_class(2)

## 保存 CSV + 看三类是否分得开

* 三类**都采到**了 → 写 `data/features.csv`（训练用的就是它）；
* 还缺类 → 写成 `data/features_partial.csv`，**不覆盖** `features.csv`（保护上一份好数据）。

重点看每类的**不同取值个数**（应该接近帧数，而不是个位数）和**亮度是否落在 10 / 67 / 109 附近**。

In [ ]:
import matplotlib.pyplot as plt

if not rows:
    raise RuntimeError("一帧都没有：先跑上面三格里的至少一格")

have = sorted(set(int(r[3]) for r in rows))
if len(have) < 3:
    csv_path = os.path.join(ROOT, "data", "features_partial.csv")
    print("!! 只采到了 %s，这份数据不完整 -> 写成 %s（不覆盖 features.csv）"
          % ("、".join(CLASS_NAME[k] for k in have), csv_path))
    print("   把缺的类补采完，再跑本格")
else:
    csv_path = os.path.join(ROOT, "data", "features.csv")

features_io.write_csv(csv_path, rows)
print("已保存", csv_path, " 共", len(rows), "行")

arr = np.array([[r[0], r[1], r[2]] for r in rows], dtype=float)
lab = np.array([r[3] for r in rows])
for k in (0, 1, 2):
    v = arr[lab == k]
    if len(v) == 0:
        print("  label%d（%s）一帧都没有" % (k, CLASS_NAME[k]))
        continue
    print("  label%d %-6s n=%3d 不同取值=%s f均值=(%6.1f,%6.1f,%6.1f) 约亮度=%.0f"
          % (k, CLASS_NAME[k], len(v),
             tuple(len(set(v[:, c])) for c in range(3)),
             v[:, 0].mean(), v[:, 1].mean(), v[:, 2].mean(),
             v[:, 0].mean() * LUMA_PER_F_LUMA))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c, name in zip(axes, range(3), list(features_io.FEATURE_COLUMNS)):
    for k in (0, 1, 2):
        v = arr[lab == k, c]
        if len(v):
            ax.hist(v, bins=25, alpha=0.5, label="label %d" % k)
    ax.set_title(name)
    ax.legend()
plt.tight_layout()
plt.show()

## 体检（闸门）

`tools/check_features.py` 按 C1~C7 给结论。三类中心对不上目标（C4）、
某类只有几个不同取值（C2）、或类之间隔着一大段没有样本的真空（C5）都会点出来。

**有 FAIL 就重采那一类，别带着这份数据去重训。**

In [ ]:
import subprocess

r = subprocess.run([sys.executable, os.path.join(ROOT, "tools", "check_features.py"),
                    csv_path])
print("check_features 退出码 =", r.returncode)
assert r.returncode == 0, "体检没通过：按上面的 FAIL 提示把那一类重采，然后再跑这一格"

## 采集完成后

1. 把 `data/features.csv` 拷回 PC 备份；
2. 重训：`03_mlp_train.ipynb` → `04_mlp_quant_export.ipynb`
   （已经用上 `train_for_quantization`，会自动躲开「w1 量化饱和」）；
3. 权重变了必须重出 bit：`make_tv.py` → HLS → `bash build_overlay.sh 12`；
4. 把这次的曝光/增益/灯位记进 `docs/04_demo_box.md`，演示当天照这个复现。

In [ ]:
# 收尾：停 VDMA；要恢复自动曝光就再调一次 exposure.set_auto(iic)。
# 残留的 DMA 会继续写 DDR，换 overlay / 关 notebook 之前一定要收尾。
vs.stop_all_channels(vdma0, vdma1)
exposure.set_auto(iic)
print("VDMA 已释放，曝光已恢复自动")